# Practice 4 · Normalization and Residual Connections

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |

### I. Background

A deep network that will not train can often be rescued by three mitigation methods: normalization,
skip connections and gradient clipping. In this practice you apply each of them **once** and compare
the **magnitude** of the gradients. No long training is needed - one forward and one backward pass
is enough to see the difference.

### II. Task Requirements

1. Set `SID` and load a small batch of CIFAR10.
2. Measure the baseline network (8 hidden layers of 64 units, sigmoid) and record the gradient norm
 of every layer.
3. Add BatchNorm and measure again, then compare the layer-wise magnitudes with the baseline.
4. Add a skip connection (a residual block) and measure again.
5. Clip the gradients and check the total magnitude stays below the clip value.

Steps: Open the notebook in Haima, fill in the missing code, and run it.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| `gn_base` | baseline layer-wise gradient norms | one value per layer |
| `gn_bn` | with BatchNorm | larger than `gn_base` |
| `net_res`, `gn_res` | the residual network and its measurement | same shapes, gradients still flow |
| `total_after`, `clip_max` | after clipping | `total_after <= clip_max` |


## Question 1 · BatchNorm

Measure the baseline first, then insert `nn.BatchNorm1d` after every hidden layer and measure again.

In [ ]:
import math, os, torch, torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]

from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader

# 🟢 EDIT: set SID to the last two digits of your student ID
SID = 23
torch.manual_seed(1000 + SID)

# 🟡 READ: one batch is enough; we never train here
# 🟡 READ: CIFAR10 ships with the platform; the check below only avoids an accidental 170 MB download
DATA_ROOT = "../practice2/data"
_cached = os.path.exists(os.path.join(DATA_ROOT, "cifar-10-batches-py"))
if not _cached:
    print("CIFAR10 not found under %s - it will be downloaded (about 170 MB), please wait ..." % DATA_ROOT)
ds = CIFAR10(DATA_ROOT, train=True, download=not _cached, transform=transforms.ToTensor())
dl = DataLoader(ds, batch_size=256, shuffle=False)
xb, yb = next(iter(dl))


def make_net(n_layer=8, act=nn.Sigmoid, bn=False):
    """MLP with n_layer hidden layers of 64 units; bn=True inserts BatchNorm1d."""
    layers, d = [nn.Flatten()], 3072
    for _ in range(n_layer):
        layers.append(nn.Linear(d, 64))
        if bn:
            layers.append(nn.BatchNorm1d(64))
        layers.append(act())
        d = 64
    layers.append(nn.Linear(d, 10))
    return nn.Sequential(*layers)


def mags(net):
    net.zero_grad()
    nn.CrossEntropyLoss()(net(xb), yb).backward()
    return [p.grad.norm().item() for p in net.parameters() if p.grad is not None and p.dim() > 1]


gn_base = mags(make_net())
# 🟢 EDIT: build the same network with BatchNorm and measure it
#   1) make_net takes bn=True, which inserts BatchNorm1d after every hidden layer -> net_bn
#   2) measure it exactly like the baseline -> gn_bn
# YOUR CODE HERE
net_bn = make_net(bn=True)
gn_bn = mags(net_bn)

print("baseline first layer : %.3e" % gn_base[0])
print("+BatchNorm first layer: %.3e" % gn_bn[0])
# 🟡 READ: this is what you report - the gradient norm of every layer, for both networks
print("%-6s %-14s %-14s" % ("layer", "baseline", "+BatchNorm"))
for k, (a, b) in enumerate(zip(gn_base, gn_bn), 1):
    print("%-6d %-14.3e %-14.3e" % (k, a, b))

print("improvement factor    : %.1fx" % (gn_bn[0] / gn_base[0]))

baseline first layer : 5.652e-07
+BatchNorm first layer: 1.119e+00
layer  baseline       +BatchNorm    
1      5.652e-07      1.119e+00     
2      7.308e-07      1.808e-01     
3      4.748e-06      1.818e-01     
4      3.575e-05      1.868e-01     
5      2.518e-04      2.012e-01     
6      1.558e-03      1.721e-01     
7      1.097e-02      1.643e-01     
8      7.178e-02      1.520e-01     
9      4.860e-01      4.750e-01     
improvement factor    : 1979538.9x


## Question 2 · A skip connection

Implement a residual block: `y = f(x) + x`, where `f` keeps the width unchanged.
Then stack a few blocks and measure again.

In [2]:
class ResBlock(nn.Module):
    """y = f(x) + x ; the width stays 64 so that the addition works."""
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(64, 64), nn.Sigmoid(), nn.Linear(64, 64))

    def forward(self, x):
        # 🟢 EDIT: one line - add the input to the output of f
        #   element-wise addition of x and f(x); both have 64 columns (see the class docstring),
        #   and this shortcut gives the gradient a direct path back to the early layers
        # YOUR CODE HERE
        return self.f(x) + x


class ResNet(nn.Module):
    def __init__(self, n_block=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Flatten(), nn.Linear(3072, 64), nn.Sigmoid())
        self.blocks = nn.Sequential(*[ResBlock() for _ in range(n_block)])
        self.head = nn.Linear(64, 10)

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))


net_res = ResNet()
# 🟢 EDIT: measure the residual network the same way as before -> gn_res
# YOUR CODE HERE
gn_res = mags(net_res)

print("baseline first layer   : %.3e" % gn_base[0])
print("+skip connection       : %.3e" % gn_res[0])
# 🟡 READ: layer-wise comparison of all three settings
print("%-6s %-14s %-14s %-14s" % ("layer", "baseline", "+BatchNorm", "+skip"))
for k, (a, b, c) in enumerate(zip(gn_base, gn_bn, gn_res), 1):
    print("%-6d %-14.3e %-14.3e %-14.3e" % (k, a, b, c))

print("output shape unchanged :", tuple(net_res(xb).shape), "==", tuple(xb.shape[:1]) + (10,))

baseline first layer   : 5.652e-07
+skip connection       : 6.307e-01
layer  baseline       +BatchNorm     +skip         
1      5.652e-07      1.119e+00      6.307e-01     
2      7.308e-07      1.808e-01      5.826e-02     
3      4.748e-06      1.818e-01      3.775e-01     
4      3.575e-05      1.868e-01      5.591e-02     
5      2.518e-04      2.012e-01      3.786e-01     
6      1.558e-03      1.721e-01      5.287e-02     
7      1.097e-02      1.643e-01      3.810e-01     
8      7.178e-02      1.520e-01      6.663e-02     
9      4.860e-01      4.750e-01      3.686e-01     
output shape unchanged : (256, 10) == (256, 10)


## Question 3 · Gradient clipping

Gradient norm clipping preserves the direction of the gradient and limits its total magnitude.
Fill in the clipping call and check the result.

If the total gradient norm is already below the threshold, clipping leaves it unchanged. With SID=42 in this run, the norm was about 0.317, below clip_max=0.5.


In [3]:
# 🟡 READ: 0.5 is small on purpose, so that the effect is visible
clip_max = 0.5

net_clip = make_net()
net_clip.zero_grad()
nn.CrossEntropyLoss()(net_clip(xb), yb).backward()

total_before = torch.nn.utils.clip_grad_norm_(net_clip.parameters(), 1e9).item()
# 🟢 EDIT: clip with clip_max and record the total magnitude after clipping
#   1) clip the gradients with torch.nn.utils.clip_grad_norm_(...) using clip_max.
#      It rescales ALL gradients together so that their GLOBAL L2 norm is at most clip_max,
#      keeping the direction of the gradient.
#   2) NOTE: that call RETURNS the global norm measured BEFORE clipping — that is why the
#      line above passes 1e9 (it just reads the un-clipped value).
#   3) So the value after clipping cannot come from the return value: recompute it yourself
#      as sqrt( sum of ||p.grad||^2 ) over every parameter that has a gradient
#      (biases included here, unlike Q1) -> total_after
# YOUR CODE HERE
torch.nn.utils.clip_grad_norm_(net_clip.parameters(), clip_max)
total_after = math.sqrt(
    sum(p.grad.norm().item() ** 2 for p in net_clip.parameters() if p.grad is not None)
)

print("total magnitude before clipping: %.4f" % total_before)
print("total magnitude after  clipping: %.4f  (limit %.2f)" % (total_after, clip_max))

total magnitude before clipping: 0.3170
total magnitude after  clipping: 0.3170  (limit 0.50)


## Question 4 · After inserting BatchNorm, the gradient of the early layers …

- (A) becomes larger, so the vanishing problem is relieved
- (B) becomes smaller      (C) stays the same


A. In this experiment, BatchNorm makes the early-layer gradients larger, relieving the vanishing-gradient problem.

## Question 5 · Why does adding a skip connection help the early layers?

- (A) it reduces the number of parameters
- (B) it gives the gradient a direct path back to the early layers
- (C) it replaces the activation function


B. The skip connection gives the gradient a direct path back to the early layers through the identity branch of y = f(x) + x.